In [8]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import BaseDocTemplate, Frame, PageTemplate, Paragraph, Spacer, Table, TableStyle, KeepTogether
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.enums import TA_LEFT, TA_CENTER
import os, math

# Register a clean font (DejaVu is usually available)
font_paths = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
]
for p in font_paths:
    if not os.path.exists(p):
        print("Missing font:", p)

pdfmetrics.registerFont(TTFont("DejaVu", font_paths[0]))
pdfmetrics.registerFont(TTFont("DejaVuBold", font_paths[1]))

styles = getSampleStyleSheet()
base = ParagraphStyle(
    "Base",
    parent=styles["Normal"],
    fontName="DejaVu",
    fontSize=9,
    leading=11,
    spaceAfter=2,
)
title = ParagraphStyle(
    "Title",
    parent=base,
    fontName="DejaVuBold",
    fontSize=16,
    leading=18,
    alignment=TA_CENTER,
    spaceAfter=6,
)
h = ParagraphStyle(
    "H",
    parent=base,
    fontName="DejaVuBold",
    fontSize=11,
    leading=13,
    spaceBefore=4,
    spaceAfter=3,
)
subh = ParagraphStyle(
    "SubH",
    parent=base,
    fontName="DejaVuBold",
    fontSize=9.5,
    leading=11,
    spaceBefore=3,
    spaceAfter=2,
)
small = ParagraphStyle(
    "Small",
    parent=base,
    fontSize=8.2,
    leading=9.6,
)

accent = colors.HexColor("#1f4e79")
light_accent = colors.HexColor("#e8f0fb")
grey = colors.HexColor("#444444")

def header_footer(canvas, doc):
    canvas.saveState()
    w, hgt = A4
    # Header bar
    canvas.setFillColor(light_accent)
    canvas.rect(0, hgt-18*mm, w, 18*mm, stroke=0, fill=1)
    canvas.setFillColor(accent)
    canvas.setFont("DejaVuBold", 11)
    canvas.drawString(12*mm, hgt-11.5*mm, "Naamgeving organische moleculen (C1–C5) — snelkaart")
    canvas.setFont("DejaVu", 8)
    canvas.setFillColor(grey)
    canvas.drawRightString(w-12*mm, hgt-11.5*mm, "IUPAC-basis: alkanen/alkenen/alkynen, halogeenalkanen, alcoholen, carbonzuren (1 functie)")
    # Footer
    canvas.setFillColor(grey)
    canvas.setFont("DejaVu", 8)
    canvas.drawString(12*mm, 8*mm, "Tip: oefen in 2 richtingen: structuur → naam én naam → structuur.")
    canvas.drawRightString(w-12*mm, 8*mm, f"Pagina {doc.page}")
    canvas.restoreState()

# Document setup with two columns
file_path = "/mnt/data/A4_Cheatsheet_Naamgeving_C1-C5.pdf"
doc = BaseDocTemplate(
    file_path,
    pagesize=A4,
    leftMargin=12*mm,
    rightMargin=12*mm,
    topMargin=22*mm,   # leave for header bar
    bottomMargin=14*mm # leave for footer
)

w, hgt = A4
usable_w = w - doc.leftMargin - doc.rightMargin
gutter = 6*mm
col_w = (usable_w - gutter) / 2
frame_h = hgt - doc.topMargin - doc.bottomMargin

frame1 = Frame(doc.leftMargin, doc.bottomMargin, col_w, frame_h, id="col1", showBoundary=0)
frame2 = Frame(doc.leftMargin + col_w + gutter, doc.bottomMargin, col_w, frame_h, id="col2", showBoundary=0)
doc.addPageTemplates([PageTemplate(id="TwoCol", frames=[frame1, frame2], onPage=header_footer)])

story = []

# Top title (within flow)
story.append(Spacer(1, 2))
story.append(Paragraph("Cheat sheet + 25 oefenitems (met oplossingen)", title))

# Section 1: mini-woordenboek
story.append(Paragraph("1) Mini-woordenboek", h))
story.append(Paragraph("<b>Stammen (hoofdketen)</b>: meth- (1), eth- (2), prop- (3), but- (4), pent- (5).", base))
story.append(Paragraph("<b>Verzadiging</b>: -aan (enkel), -een (dubbel), -yn (drievoudig). Locant bij de binding: but-1-een, pent-2-yn.", base))
story.append(Paragraph("<b>Substituenten</b>: methyl- (CH3), ethyl- (C2H5); halogenen: fluor-, chloor-, broom-, jood-. Meervoud: di-, tri-.", base))
story.append(Paragraph("<b>Functionele groep (1 per molecule)</b>: alcohol = -ol; carbonzuur = -zuur.", base))

# Section 2: prioriteit & nummering
story.append(Paragraph("2) Prioriteit (wat krijgt het laagste nummer?)", h))
story.append(Paragraph("carbonzuur (COOH) &gt; alcohol (OH) &gt; dubbele/drievoudige binding &gt; substituenten (halogeen/alkyl).", base))
story.append(Paragraph("Nummer de hoofdketen zodat het <b>eerste verschil</b> zo klein mogelijk is (lowest set of locants).", base))

# Section 3: stappenplan
story.append(Paragraph("3) Stappenplan (structuur → naam)", h))
steps = [
    "Kies de <b>langste keten</b> die de functie (COOH/OH) en/of meervoudige binding bevat.",
    "Bepaal het <b>suffix</b>: -zuur, -ol, -een, -yn, -aan (in die volgorde).",
    "Nummer voor laagste locant van: functie → binding → substituenten.",
    "Schrijf substituenten met locanten, <b>alfabetisch</b> (di/tri tellen niet mee voor alfabet).",
    "Combineer: <i>locanten–prefixen + stam + locant binding + suffix</i>."
]
story.append(Paragraph("<br/>".join([f"• {s}" for s in steps]), base))

# Section 4: veelgemaakte fouten
story.append(Paragraph("4) Klassieke valkuilen", h))
pitfalls = [
    "S verkeerd nummeren: bij alcohol/zuur moet de C met functie zo laag mogelijk.",
    "Hoofdketen te kort kiezen: neem echt de langste (maar moet de functie/binding bevatten).",
    "Verwarren van isomeren: 1-chloorpropaan ≠ 2-chloorpropaan; but-1-een ≠ but-2-een.",
    "Prefixen niet alfabetisch: broom- komt vóór chloor-; methyl- komt na jood-.",
]
story.append(Paragraph("<br/>".join([f"• {p}" for p in pitfalls]), base))

# Exercises table (two halves to fit)
story.append(Paragraph("5) Oefenen (25) — antwoorden rechts", h))
# Prepare 25 items: mixture of name-from-structure and structure-from-name
ex = [
 ("CH3–CH2–CH3", "propaan"),
 ("CH3–CH2–CH2–CH3", "butaan"),
 ("CH3–CH(CH3)–CH3", "2-methylpropaan"),
 ("CH3–CH2–CH2–CH2–CH3", "pentaan"),
 ("CH3–C(CH3)2–CH3", "2,2-dimethylpropaan"),
 ("CH2=CH–CH3", "prop-1-een (propeen)"),
 ("CH3–CH=CH–CH3", "but-2-een"),
 ("CH≡C–CH3", "prop-1-yn (propyn)"),
 ("CH3–CH2–C≡CH", "but-1-yn"),
 ("CH3–CH2–CH(OH)–CH3", "butan-2-ol"),
 ("CH3–CH(OH)–CH3", "propan-2-ol (isopropanol)"),
 ("HO–CH2–CH2–CH3", "propan-1-ol"),
 ("CH3–CH2–COOH", "propaanzuur"),
 ("CH3–COOH", "ethaanzuur (azijnzuur)"),
 ("(CH3)2CH–COOH", "2-methylpropaanzuur"),
 ("CH3–CH2–CH2–Cl", "1-chloorpropaan"),
 ("CH3–CH(Cl)–CH3", "2-chloorpropaan"),
 ("CH3–CH2–CH(Br)–CH3", "2-broombutaan"),
 ("CH3–CH(Cl)–CH2–CH3", "2-chloorbutaan"),
 ("CH3–CH2–CH2–CH2–I", "1-joodbutaan"),
 ("1,2-dichloorethaan", "Cl–CH2–CH2–Cl"),
 ("1,1-dichloorethaan", "CH3–CHCl2"),
 ("2-methylbutaan", "CH3–CH(CH3)–CH2–CH3"),
 ("pent-2-een", "CH3–CH=CH–CH2–CH3"),
 ("2,3-dibroompentaan", "CH3–CH(Br)–CH(Br)–CH2–CH3"),
]

# Split into two tables of 13 and 12 rows to control height
def make_table(items, font=7.9):
    data = [["Oefening", "Antwoord / structuur"]]
    for q, a in items:
        data.append([q, a])
    tbl = Table(data, colWidths=[col_w*0.52, col_w*0.48])
    tbl.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,0), "DejaVuBold"),
        ("FONTSIZE", (0,0), (-1,0), 8.2),
        ("BACKGROUND", (0,0), (-1,0), light_accent),
        ("TEXTCOLOR", (0,0), (-1,0), accent),
        ("ALIGN", (0,0), (-1,0), "LEFT"),
        ("FONTNAME", (0,1), (-1,-1), "DejaVu"),
        ("FONTSIZE", (0,1), (-1,-1), font),
        ("LEADING", (0,1), (-1,-1), font+1.2),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.whitesmoke, colors.white]),
        ("GRID", (0,0), (-1,-1), 0.25, colors.lightgrey),
        ("LEFTPADDING", (0,0), (-1,-1), 4),
        ("RIGHTPADDING", (0,0), (-1,-1), 4),
        ("TOPPADDING", (0,0), (-1,-1), 2),
        ("BOTTOMPADDING", (0,0), (-1,-1), 2),
    ]))
    return tbl

tbl1 = make_table(ex[:13])
tbl2 = make_table(ex[13:])

story.append(KeepTogether([tbl1, Spacer(1, 6), Paragraph("Extra tip: bij naam → structuur: teken eerst de hoofdketen (stam + binding), nummer, voeg substituenten toe, zet functie op de juiste C.", small)]))
story.append(Spacer(1, 6))
story.append(tbl2)

doc.build(story)

file_path


Missing font: /usr/share/fonts/truetype/dejavu/DejaVuSans.ttf
Missing font: /usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf


TTFError: Can't open file "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"

In [6]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import BaseDocTemplate, Frame, PageTemplate, Paragraph, Spacer, Table, TableStyle, FrameBreak, KeepTogether
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.enums import TA_LEFT, TA_CENTER
import os, math

# Fonts
pdfmetrics.registerFont(TTFont("DejaVu", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"))
pdfmetrics.registerFont(TTFont("DejaVuBold", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"))

styles = getSampleStyleSheet()
base = ParagraphStyle(
    "Base",
    parent=styles["Normal"],
    fontName="DejaVu",
    fontSize=9,
    leading=11,
    spaceAfter=2,
)
title = ParagraphStyle(
    "Title",
    parent=base,
    fontName="DejaVuBold",
    fontSize=14,
    leading=16,
    alignment=TA_LEFT,
    spaceAfter=6,
    textColor=colors.HexColor("#0b2f4f"),
)
h = ParagraphStyle(
    "H",
    parent=base,
    fontName="DejaVuBold",
    fontSize=10.6,
    leading=12.5,
    spaceBefore=4,
    spaceAfter=3,
    textColor=colors.HexColor("#0b2f4f"),
)
small = ParagraphStyle(
    "Small",
    parent=base,
    fontSize=8.1,
    leading=9.6,
)

accent = colors.HexColor("#1f4e79")
light_accent = colors.HexColor("#e8f0fb")
grey = colors.HexColor("#444444")

def header_footer(canvas, doc):
    canvas.saveState()
    w, hgt = A4
    # Header bar
    bar_h = 20*mm
    canvas.setFillColor(light_accent)
    canvas.rect(0, hgt-bar_h, w, bar_h, stroke=0, fill=1)
    canvas.setFillColor(accent)
    canvas.setFont("DejaVuBold", 11)
    canvas.drawString(12*mm, hgt-8.5*mm, "Naamgeving organische moleculen (C1–C5)")
    canvas.setFont("DejaVu", 8)
    canvas.setFillColor(grey)
    canvas.drawString(12*mm, hgt-15.5*mm, "Koolwaterstoffen, halogeenalkanen, alcoholen, carbonzuren (1 functionele groep)")
    # Footer
    canvas.setFillColor(grey)
    canvas.setFont("DejaVu", 8)
    canvas.drawString(12*mm, 8*mm, "Drill: structuur → naam én naam → structuur. Focus op locanten en prioriteit.")
    canvas.drawRightString(w-12*mm, 8*mm, f"Pagina {doc.page}")
    canvas.restoreState()

file_path = "/mnt/data/A4_Cheatsheet_Naamgeving_C1-C5.pdf"
doc = BaseDocTemplate(
    file_path,
    pagesize=A4,
    leftMargin=12*mm,
    rightMargin=12*mm,
    topMargin=24*mm,   # leave for header bar
    bottomMargin=14*mm
)

w, hgt = A4
usable_w = w - doc.leftMargin - doc.rightMargin
gutter = 6*mm
col_w = (usable_w - gutter) / 2
frame_h = hgt - doc.topMargin - doc.bottomMargin

frame1 = Frame(doc.leftMargin, doc.bottomMargin, col_w, frame_h, id="col1", showBoundary=0)
frame2 = Frame(doc.leftMargin + col_w + gutter, doc.bottomMargin, col_w, frame_h, id="col2", showBoundary=0)
doc.addPageTemplates([PageTemplate(id="TwoCol", frames=[frame1, frame2], onPage=header_footer)])

story = []
story.append(Spacer(1, 2))
story.append(Paragraph("Cheat sheet + 25 oefenitems (met oplossingen)", title))

# Left column content
story.append(Paragraph("1) Mini-woordenboek", h))
story.append(Paragraph("<b>Stammen</b>: meth- (1), eth- (2), prop- (3), but- (4), pent- (5).", base))
story.append(Paragraph("<b>Verzadiging</b>: -aan (enkel), -een (dubbel), -yn (drievoudig). Locant bij binding: but-1-een, pent-2-yn.", base))
story.append(Paragraph("<b>Substituenten</b>: methyl- (CH3), ethyl- (C2H5); halogenen: fluor-, chloor-, broom-, jood-. Meervoud: di-, tri-.", base))
story.append(Paragraph("<b>Functie (1 per molecule)</b>: alcohol = -ol; carbonzuur = -zuur.", base))

story.append(Paragraph("2) Prioriteit en nummering", h))
story.append(Paragraph("<b>carbonzuur</b> (COOH) &gt; <b>alcohol</b> (OH) &gt; dubbele/drievoudige binding &gt; substituenten.", base))
story.append(Paragraph("Nummer zodat de <b>laagste locanten</b> ontstaan (first point of difference).", base))

story.append(Paragraph("3) Stappenplan (structuur → naam)", h))
steps = [
    "Kies de <b>langste keten</b> die de functie/binding bevat.",
    "Kies het <b>suffix</b>: -zuur, -ol, -een, -yn, -aan.",
    "Nummer: functie → binding → substituenten.",
    "Schrijf substituenten met locanten, <b>alfabetisch</b> (di/tri tellen niet mee).",
    "Combineer: locanten–prefixen + stam + (locant binding) + suffix."
]
story.append(Paragraph("<br/>".join([f"• {s}" for s in steps]), base))

story.append(Paragraph("4) Klassieke valkuilen", h))
pitfalls = [
    "S is absoluut; <b>niet</b> verwarren met vormingsgrootheden (maar voor naamgeving: let op locanten!).",
    "Hoofdketen te kort: neem echt de langste die de functie/binding bevat.",
    "Isomeren: 1-chloorpropaan ≠ 2-chloorpropaan; but-1-een ≠ but-2-een.",
    "Prefixen alfabetisch: broom- vóór chloor-; methyl- komt na jood-."
]
story.append(Paragraph("<br/>".join([f"• {p}" for p in pitfalls]), base))

story.append(Paragraph("5) Ankers (ken deze blind)", h))
anchors = [
    "<b>C3</b>: propaan; prop-1-een; prop-1-yn; propan-1-ol; propan-2-ol; propaanzuur.",
    "<b>C4</b>: butaan; 2-methylpropaan; but-1-een; but-2-een; butan-1-ol; butan-2-ol; butaanzuur.",
    "<b>C5</b>: pentaan; 2-methylbutaan; 2,2-dimethylpropaan; pent-2-een; pentaanzuur.",
]
story.append(Paragraph("<br/>".join([f"• {a}" for a in anchors]), base))

story.append(Paragraph("6) Minimal pairs (snelle check)", h))
mp = [
    "propan-1-ol vs propan-2-ol",
    "1,1-dichloorethaan vs 1,2-dichloorethaan",
    "but-1-een vs but-2-een",
    "2-methylbutaan vs 2,2-dimethylpropaan"
]
story.append(Paragraph("<br/>".join([f"• {m}" for m in mp]), base))

# Force next column for exercises
story.append(FrameBreak())

# Exercises right column
story.append(Paragraph("7) Oefenen (25) — met oplossingen", h))
story.append(Paragraph("Mengvorm: soms structuur → naam, soms naam → structuur. Antwoord staat rechts.", small))
story.append(Spacer(1, 4))

ex = [
 ("CH3–CH2–CH3", "propaan"),
 ("CH3–CH2–CH2–CH3", "butaan"),
 ("CH3–CH(CH3)–CH3", "2-methylpropaan"),
 ("CH3–CH2–CH2–CH2–CH3", "pentaan"),
 ("CH3–C(CH3)2–CH3", "2,2-dimethylpropaan"),
 ("CH2=CH–CH3", "prop-1-een (propeen)"),
 ("CH3–CH=CH–CH3", "but-2-een"),
 ("CH≡C–CH3", "prop-1-yn (propyn)"),
 ("CH3–CH2–C≡CH", "but-1-yn"),
 ("CH3–CH2–CH(OH)–CH3", "butan-2-ol"),
 ("CH3–CH(OH)–CH3", "propan-2-ol (isopropanol)"),
 ("HO–CH2–CH2–CH3", "propan-1-ol"),
 ("CH3–CH2–COOH", "propaanzuur"),
 ("CH3–COOH", "ethaanzuur (azijnzuur)"),
 ("(CH3)2CH–COOH", "2-methylpropaanzuur"),
 ("CH3–CH2–CH2–Cl", "1-chloorpropaan"),
 ("CH3–CH(Cl)–CH3", "2-chloorpropaan"),
 ("CH3–CH2–CH(Br)–CH3", "2-broombutaan"),
 ("CH3–CH(Cl)–CH2–CH3", "2-chloorbutaan"),
 ("CH3–CH2–CH2–CH2–I", "1-joodbutaan"),
 ("1,2-dichloorethaan", "Cl–CH2–CH2–Cl"),
 ("1,1-dichloorethaan", "CH3–CHCl2"),
 ("2-methylbutaan", "CH3–CH(CH3)–CH2–CH3"),
 ("pent-2-een", "CH3–CH=CH–CH2–CH3"),
 ("2,3-dibroompentaan", "CH3–CH(Br)–CH(Br)–CH2–CH3"),
]

def make_table(items, font=7.8):
    data = [["Oefening", "Antwoord / structuur"]]
    for q, a in items:
        data.append([q, a])
    tbl = Table(data, colWidths=[col_w*0.52, col_w*0.48])
    tbl.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,0), "DejaVuBold"),
        ("FONTSIZE", (0,0), (-1,0), 8.2),
        ("BACKGROUND", (0,0), (-1,0), light_accent),
        ("TEXTCOLOR", (0,0), (-1,0), accent),
        ("FONTNAME", (0,1), (-1,-1), "DejaVu"),
        ("FONTSIZE", (0,1), (-1,-1), font),
        ("LEADING", (0,1), (-1,-1), font+1.2),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.whitesmoke, colors.white]),
        ("GRID", (0,0), (-1,-1), 0.25, colors.lightgrey),
        ("LEFTPADDING", (0,0), (-1,-1), 4),
        ("RIGHTPADDING", (0,0), (-1,-1), 4),
        ("TOPPADDING", (0,0), (-1,-1), 2),
        ("BOTTOMPADDING", (0,0), (-1,-1), 2),
    ]))
    return tbl

tbl1 = make_table(ex[:13])
tbl2 = make_table(ex[13:])

story.append(tbl1)
story.append(Spacer(1, 6))
story.append(tbl2)
story.append(Spacer(1, 6))
story.append(Paragraph("<b>Bonus:</b> maak zelf 5 extra kaarten met minimale paren uit jouw oefenzittingen.", small))

doc.build(story)

file_path


TTFError: Can't open file "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"

In [3]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import BaseDocTemplate, Frame, PageTemplate, Paragraph, Spacer, Table, TableStyle, FrameBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.enums import TA_LEFT

# Fonts
pdfmetrics.registerFont(TTFont("DejaVu", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"))
pdfmetrics.registerFont(TTFont("DejaVuBold", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"))

styles = getSampleStyleSheet()
base = ParagraphStyle("Base", parent=styles["Normal"], fontName="DejaVu", fontSize=9, leading=11, spaceAfter=2)
title = ParagraphStyle("Title", parent=base, fontName="DejaVuBold", fontSize=14, leading=16, alignment=TA_LEFT, spaceAfter=6, textColor=colors.HexColor("#0b2f4f"))
h = ParagraphStyle("H", parent=base, fontName="DejaVuBold", fontSize=10.6, leading=12.5, spaceBefore=4, spaceAfter=3, textColor=colors.HexColor("#0b2f4f"))
small = ParagraphStyle("Small", parent=base, fontSize=8.1, leading=9.6)

accent = colors.HexColor("#1f4e79")
light_accent = colors.HexColor("#e8f0fb")
grey = colors.HexColor("#444444")

def header_footer(canvas, doc):
    canvas.saveState()
    w, hgt = A4
    bar_h = 20*mm
    canvas.setFillColor(light_accent)
    canvas.rect(0, hgt-bar_h, w, bar_h, stroke=0, fill=1)
    canvas.setFillColor(accent)
    canvas.setFont("DejaVuBold", 11)
    canvas.drawString(12*mm, hgt-8.5*mm, "Naamgeving organische moleculen (C1–C5)")
    canvas.setFont("DejaVu", 8)
    canvas.setFillColor(grey)
    canvas.drawString(12*mm, hgt-15.5*mm, "Koolwaterstoffen, halogeenalkanen, alcoholen, carbonzuren (1 functionele groep)")
    canvas.setFillColor(grey)
    canvas.setFont("DejaVu", 8)
    canvas.drawString(12*mm, 8*mm, "Drill: structuur → naam én naam → structuur. Focus op locanten en prioriteit.")
    canvas.drawRightString(w-12*mm, 8*mm, f"Pagina {doc.page}")
    canvas.restoreState()

file_path = "/mnt/data/A4_Cheatsheet_Naamgeving_C1-C5.pdf"
doc = BaseDocTemplate(file_path, pagesize=A4, leftMargin=12*mm, rightMargin=12*mm, topMargin=24*mm, bottomMargin=14*mm)

w, hgt = A4
usable_w = w - doc.leftMargin - doc.rightMargin
gutter = 6*mm
col_w = (usable_w - gutter) / 2
frame_h = hgt - doc.topMargin - doc.bottomMargin
frame1 = Frame(doc.leftMargin, doc.bottomMargin, col_w, frame_h, id="col1", showBoundary=0)
frame2 = Frame(doc.leftMargin + col_w + gutter, doc.bottomMargin, col_w, frame_h, id="col2", showBoundary=0)
doc.addPageTemplates([PageTemplate(id="TwoCol", frames=[frame1, frame2], onPage=header_footer)])

story = [Spacer(1,2), Paragraph("Cheat sheet + 25 oefenitems (met oplossingen)", title)]

story.append(Paragraph("1) Mini-woordenboek", h))
story.append(Paragraph("<b>Stammen</b>: meth- (1), eth- (2), prop- (3), but- (4), pent- (5).", base))
story.append(Paragraph("<b>Verzadiging</b>: -aan (enkel), -een (dubbel), -yn (drievoudig). Locant bij binding: but-1-een, pent-2-yn.", base))
story.append(Paragraph("<b>Substituenten</b>: methyl- (CH3), ethyl- (C2H5); halogenen: fluor-, chloor-, broom-, jood-. Meervoud: di-, tri-.", base))
story.append(Paragraph("<b>Functie (1 per molecule)</b>: alcohol = -ol; carbonzuur = -zuur.", base))

story.append(Paragraph("2) Prioriteit en nummering", h))
story.append(Paragraph("<b>carbonzuur</b> (COOH) &gt; <b>alcohol</b> (OH) &gt; dubbele/drievoudige binding &gt; substituenten.", base))
story.append(Paragraph("Nummer zodat de <b>laagste locanten</b> ontstaan (first point of difference).", base))

story.append(Paragraph("3) Stappenplan (structuur → naam)", h))
steps = [
    "Kies de <b>langste keten</b> die de functie/binding bevat.",
    "Kies het <b>suffix</b>: -zuur, -ol, -een, -yn, -aan.",
    "Nummer: functie → binding → substituenten.",
    "Schrijf substituenten met locanten, <b>alfabetisch</b> (di/tri tellen niet mee).",
    "Combineer: locanten–prefixen + stam + (locant binding) + suffix."
]
story.append(Paragraph("<br/>".join([f"• {s}" for s in steps]), base))

story.append(Paragraph("4) Klassieke valkuilen", h))
pitfalls = [
    "Di-/tri- <b>niet</b> meetellen voor alfabet (dichloor- blijft onder 'chloor').",
    "Hoofdketen te kort: neem echt de langste die de functie/binding bevat.",
    "Isomeren: 1-chloorpropaan ≠ 2-chloorpropaan; but-1-een ≠ but-2-een.",
    "Locanten vergeten bij alkenen/alkynen: but-1-een is iets anders dan but-2-een."
]
story.append(Paragraph("<br/>".join([f"• {p}" for p in pitfalls]), base))

story.append(Paragraph("5) Ankers (ken deze blind)", h))
anchors = [
    "<b>C3</b>: propaan; prop-1-een; prop-1-yn; propan-1-ol; propan-2-ol; propaanzuur.",
    "<b>C4</b>: butaan; 2-methylpropaan; but-1-een; but-2-een; butan-1-ol; butan-2-ol; butaanzuur.",
    "<b>C5</b>: pentaan; 2-methylbutaan; 2,2-dimethylpropaan; pent-2-een; pentaanzuur.",
]
story.append(Paragraph("<br/>".join([f"• {a}" for a in anchors]), base))

story.append(Paragraph("6) Minimal pairs (snelle check)", h))
mp = [
    "propan-1-ol vs propan-2-ol",
    "1,1-dichloorethaan vs 1,2-dichloorethaan",
    "but-1-een vs but-2-een",
    "2-methylbutaan vs 2,2-dimethylpropaan"
]
story.append(Paragraph("<br/>".join([f"• {m}" for m in mp]), base))

story.append(FrameBreak())

story.append(Paragraph("7) Oefenen (25) — met oplossingen", h))
story.append(Paragraph("Mengvorm: soms structuur → naam, soms naam → structuur. Antwoord staat rechts.", small))
story.append(Spacer(1,4))

ex = [
 ("CH3–CH2–CH3", "propaan"),
 ("CH3–CH2–CH2–CH3", "butaan"),
 ("CH3–CH(CH3)–CH3", "2-methylpropaan"),
 ("CH3–CH2–CH2–CH2–CH3", "pentaan"),
 ("CH3–C(CH3)2–CH3", "2,2-dimethylpropaan"),
 ("CH2=CH–CH3", "prop-1-een (propeen)"),
 ("CH3–CH=CH–CH3", "but-2-een"),
 ("CH≡C–CH3", "prop-1-yn (propyn)"),
 ("CH3–CH2–C≡CH", "but-1-yn"),
 ("CH3–CH2–CH(OH)–CH3", "butan-2-ol"),
 ("CH3–CH(OH)–CH3", "propan-2-ol (isopropanol)"),
 ("HO–CH2–CH2–CH3", "propan-1-ol"),
 ("CH3–CH2–COOH", "propaanzuur"),
 ("CH3–COOH", "ethaanzuur (azijnzuur)"),
 ("(CH3)2CH–COOH", "2-methylpropaanzuur"),
 ("CH3–CH2–CH2–Cl", "1-chloorpropaan"),
 ("CH3–CH(Cl)–CH3", "2-chloorpropaan"),
 ("CH3–CH2–CH(Br)–CH3", "2-broombutaan"),
 ("CH3–CH(Cl)–CH2–CH3", "2-chloorbutaan"),
 ("CH3–CH2–CH2–CH2–I", "1-joodbutaan"),
 ("1,2-dichloorethaan", "Cl–CH2–CH2–Cl"),
 ("1,1-dichloorethaan", "CH3–CHCl2"),
 ("2-methylbutaan", "CH3–CH(CH3)–CH2–CH3"),
 ("pent-2-een", "CH3–CH=CH–CH2–CH3"),
 ("2,3-dibroompentaan", "CH3–CH(Br)–CH(Br)–CH2–CH3"),
]

def make_table(items, font=7.8):
    data = [["Oefening", "Antwoord / structuur"]]
    for q, a in items:
        data.append([q, a])
    tbl = Table(data, colWidths=[col_w*0.52, col_w*0.48])
    tbl.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,0), "DejaVuBold"),
        ("FONTSIZE", (0,0), (-1,0), 8.2),
        ("BACKGROUND", (0,0), (-1,0), light_accent),
        ("TEXTCOLOR", (0,0), (-1,0), accent),
        ("FONTNAME", (0,1), (-1,-1), "DejaVu"),
        ("FONTSIZE", (0,1), (-1,-1), font),
        ("LEADING", (0,1), (-1,-1), font+1.2),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.whitesmoke, colors.white]),
        ("GRID", (0,0), (-1,-1), 0.25, colors.lightgrey),
        ("LEFTPADDING", (0,0), (-1,-1), 4),
        ("RIGHTPADDING", (0,0), (-1,-1), 4),
        ("TOPPADDING", (0,0), (-1,-1), 2),
        ("BOTTOMPADDING", (0,0), (-1,-1), 2),
    ]))
    return tbl

story.append(make_table(ex[:13]))
story.append(Spacer(1,6))
story.append(make_table(ex[13:]))
story.append(Spacer(1,6))
story.append(Paragraph("<b>Bonus:</b> voeg 5 eigen kaarten toe uit oefeningen/oud-examens.", small))

doc.build(story)
file_path


ModuleNotFoundError: No module named 'reportlab'

In [4]:
# Retry PDF build (same code, compact) 
from reportlab.lib.pagesizes import A4
from reportlab.platypus import BaseDocTemplate, Frame, PageTemplate, Paragraph, Spacer, Table, TableStyle, FrameBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.enums import TA_LEFT

pdfmetrics.registerFont(TTFont("DejaVu", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"))
pdfmetrics.registerFont(TTFont("DejaVuBold", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"))

styles = getSampleStyleSheet()
base = ParagraphStyle("Base", parent=styles["Normal"], fontName="DejaVu", fontSize=9, leading=11, spaceAfter=2)
title = ParagraphStyle("Title", parent=base, fontName="DejaVuBold", fontSize=14, leading=16, alignment=TA_LEFT, spaceAfter=6, textColor=colors.HexColor("#0b2f4f"))
h = ParagraphStyle("H", parent=base, fontName="DejaVuBold", fontSize=10.6, leading=12.5, spaceBefore=4, spaceAfter=3, textColor=colors.HexColor("#0b2f4f"))
small = ParagraphStyle("Small", parent=base, fontSize=8.1, leading=9.6)

accent = colors.HexColor("#1f4e79")
light_accent = colors.HexColor("#e8f0fb")
grey = colors.HexColor("#444444")

def header_footer(canvas, doc):
    canvas.saveState()
    w, hgt = A4
    bar_h = 20*mm
    canvas.setFillColor(light_accent)
    canvas.rect(0, hgt-bar_h, w, bar_h, stroke=0, fill=1)
    canvas.setFillColor(accent)
    canvas.setFont("DejaVuBold", 11)
    canvas.drawString(12*mm, hgt-8.5*mm, "Naamgeving organische moleculen (C1–C5)")
    canvas.setFont("DejaVu", 8)
    canvas.setFillColor(grey)
    canvas.drawString(12*mm, hgt-15.5*mm, "Koolwaterstoffen, halogeenalkanen, alcoholen, carbonzuren (1 functionele groep)")
    canvas.setFillColor(grey)
    canvas.setFont("DejaVu", 8)
    canvas.drawString(12*mm, 8*mm, "Drill: structuur → naam én naam → structuur. Focus op locanten en prioriteit.")
    canvas.drawRightString(w-12*mm, 8*mm, f"Pagina {doc.page}")
    canvas.restoreState()

file_path = "/mnt/data/A4_Cheatsheet_Naamgeving_C1-C5.pdf"
doc = BaseDocTemplate(file_path, pagesize=A4, leftMargin=12*mm, rightMargin=12*mm, topMargin=24*mm, bottomMargin=14*mm)

w, hgt = A4
usable_w = w - doc.leftMargin - doc.rightMargin
gutter = 6*mm
col_w = (usable_w - gutter) / 2
frame_h = hgt - doc.topMargin - doc.bottomMargin
frame1 = Frame(doc.leftMargin, doc.bottomMargin, col_w, frame_h, id="col1", showBoundary=0)
frame2 = Frame(doc.leftMargin + col_w + gutter, doc.bottomMargin, col_w, frame_h, id="col2", showBoundary=0)
doc.addPageTemplates([PageTemplate(id="TwoCol", frames=[frame1, frame2], onPage=header_footer)])

story = [Spacer(1,2), Paragraph("Cheat sheet + 25 oefenitems (met oplossingen)", title)]

story += [
    Paragraph("1) Mini-woordenboek", h),
    Paragraph("<b>Stammen</b>: meth- (1), eth- (2), prop- (3), but- (4), pent- (5).", base),
    Paragraph("<b>Verzadiging</b>: -aan (enkel), -een (dubbel), -yn (drievoudig). Locant bij binding: but-1-een, pent-2-yn.", base),
    Paragraph("<b>Substituenten</b>: methyl- (CH3), ethyl- (C2H5); halogenen: fluor-, chloor-, broom-, jood-. Meervoud: di-, tri-.", base),
    Paragraph("<b>Functie (1 per molecule)</b>: alcohol = -ol; carbonzuur = -zuur.", base),
    Paragraph("2) Prioriteit en nummering", h),
    Paragraph("<b>carbonzuur</b> (COOH) &gt; <b>alcohol</b> (OH) &gt; dubbele/drievoudige binding &gt; substituenten.", base),
    Paragraph("Nummer zodat de <b>laagste locanten</b> ontstaan (first point of difference).", base),
    Paragraph("3) Stappenplan (structuur → naam)", h),
]
steps = [
    "Kies de <b>langste keten</b> die de functie/binding bevat.",
    "Kies het <b>suffix</b>: -zuur, -ol, -een, -yn, -aan.",
    "Nummer: functie → binding → substituenten.",
    "Schrijf substituenten met locanten, <b>alfabetisch</b> (di/tri tellen niet mee).",
    "Combineer: locanten–prefixen + stam + (locant binding) + suffix."
]
story.append(Paragraph("<br/>".join([f"• {s}" for s in steps]), base))

story.append(Paragraph("4) Klassieke valkuilen", h))
pitfalls = [
    "Di-/tri- <b>niet</b> meetellen voor alfabet (dichloor- blijft onder 'chloor').",
    "Hoofdketen te kort: neem echt de langste die de functie/binding bevat.",
    "Isomeren: 1-chloorpropaan ≠ 2-chloorpropaan; but-1-een ≠ but-2-een.",
    "Locanten vergeten bij alkenen/alkynen: but-1-een is iets anders dan but-2-een."
]
story.append(Paragraph("<br/>".join([f"• {p}" for p in pitfalls]), base))

story.append(Paragraph("5) Ankers (ken deze blind)", h))
anchors = [
    "<b>C3</b>: propaan; prop-1-een; prop-1-yn; propan-1-ol; propan-2-ol; propaanzuur.",
    "<b>C4</b>: butaan; 2-methylpropaan; but-1-een; but-2-een; butan-1-ol; butan-2-ol; butaanzuur.",
    "<b>C5</b>: pentaan; 2-methylbutaan; 2,2-dimethylpropaan; pent-2-een; pentaanzuur.",
]
story.append(Paragraph("<br/>".join([f"• {a}" for a in anchors]), base))

story.append(Paragraph("6) Minimal pairs (snelle check)", h))
mp = [
    "propan-1-ol vs propan-2-ol",
    "1,1-dichloorethaan vs 1,2-dichloorethaan",
    "but-1-een vs but-2-een",
    "2-methylbutaan vs 2,2-dimethylpropaan"
]
story.append(Paragraph("<br/>".join([f"• {m}" for m in mp]), base))

story.append(FrameBreak())

story.append(Paragraph("7) Oefenen (25) — met oplossingen", h))
story.append(Paragraph("Mengvorm: soms structuur → naam, soms naam → structuur. Antwoord staat rechts.", small))
story.append(Spacer(1,4))

ex = [
 ("CH3–CH2–CH3", "propaan"),
 ("CH3–CH2–CH2–CH3", "butaan"),
 ("CH3–CH(CH3)–CH3", "2-methylpropaan"),
 ("CH3–CH2–CH2–CH2–CH3", "pentaan"),
 ("CH3–C(CH3)2–CH3", "2,2-dimethylpropaan"),
 ("CH2=CH–CH3", "prop-1-een (propeen)"),
 ("CH3–CH=CH–CH3", "but-2-een"),
 ("CH≡C–CH3", "prop-1-yn (propyn)"),
 ("CH3–CH2–C≡CH", "but-1-yn"),
 ("CH3–CH2–CH(OH)–CH3", "butan-2-ol"),
 ("CH3–CH(OH)–CH3", "propan-2-ol (isopropanol)"),
 ("HO–CH2–CH2–CH3", "propan-1-ol"),
 ("CH3–CH2–COOH", "propaanzuur"),
 ("CH3–COOH", "ethaanzuur (azijnzuur)"),
 ("(CH3)2CH–COOH", "2-methylpropaanzuur"),
 ("CH3–CH2–CH2–Cl", "1-chloorpropaan"),
 ("CH3–CH(Cl)–CH3", "2-chloorpropaan"),
 ("CH3–CH2–CH(Br)–CH3", "2-broombutaan"),
 ("CH3–CH(Cl)–CH2–CH3", "2-chloorbutaan"),
 ("CH3–CH2–CH2–CH2–I", "1-joodbutaan"),
 ("1,2-dichloorethaan", "Cl–CH2–CH2–Cl"),
 ("1,1-dichloorethaan", "CH3–CHCl2"),
 ("2-methylbutaan", "CH3–CH(CH3)–CH2–CH3"),
 ("pent-2-een", "CH3–CH=CH–CH2–CH3"),
 ("2,3-dibroompentaan", "CH3–CH(Br)–CH(Br)–CH2–CH3"),
]

def make_table(items, font=7.8):
    data = [["Oefening", "Antwoord / structuur"]]
    for q, a in items:
        data.append([q, a])
    tbl = Table(data, colWidths=[col_w*0.52, col_w*0.48])
    tbl.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,0), "DejaVuBold"),
        ("FONTSIZE", (0,0), (-1,0), 8.2),
        ("BACKGROUND", (0,0), (-1,0), light_accent),
        ("TEXTCOLOR", (0,0), (-1,0), accent),
        ("FONTNAME", (0,1), (-1,-1), "DejaVu"),
        ("FONTSIZE", (0,1), (-1,-1), font),
        ("LEADING", (0,1), (-1,-1), font+1.2),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.whitesmoke, colors.white]),
        ("GRID", (0,0), (-1,-1), 0.25, colors.lightgrey),
        ("LEFTPADDING", (0,0), (-1,-1), 4),
        ("RIGHTPADDING", (0,0), (-1,-1), 4),
        ("TOPPADDING", (0,0), (-1,-1), 2),
        ("BOTTOMPADDING", (0,0), (-1,-1), 2),
    ]))
    return tbl

story.append(make_table(ex[:13]))
story.append(Spacer(1,6))
story.append(make_table(ex[13:]))
story.append(Spacer(1,6))
story.append(Paragraph("<b>Bonus:</b> voeg 5 eigen kaarten toe uit oefeningen/oud-examens.", small))

doc.build(story)
file_path


ModuleNotFoundError: No module named 'reportlab'

In [9]:
import os
from pathlib import Path
from reportlab.lib.pagesizes import A4
from reportlab.platypus import BaseDocTemplate, Frame, PageTemplate, Paragraph, Spacer, Table, TableStyle, FrameBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.enums import TA_LEFT

# Fonts voor macOS
dejavu_path = "/System/Library/Fonts/DejaVuSans.ttf"
dejavu_bold_path = "/System/Library/Fonts/DejaVuSans-Bold.ttf"

# Check en registreer fonts
if os.path.exists(dejavu_path) and os.path.exists(dejavu_bold_path):
    pdfmetrics.registerFont(TTFont("DejaVu", dejavu_path))
    pdfmetrics.registerFont(TTFont("DejaVuBold", dejavu_bold_path))
    print("✓ Fonts geladen")
else:
    print("⚠ Fonts niet gevonden. Fallback naar standaard fonts.")

styles = getSampleStyleSheet()
base = ParagraphStyle("Base", parent=styles["Normal"], fontName="DejaVu", fontSize=9, leading=11, spaceAfter=2)
title = ParagraphStyle("Title", parent=base, fontName="DejaVuBold", fontSize=14, leading=16, alignment=TA_LEFT, spaceAfter=6, textColor=colors.HexColor("#0b2f4f"))
h = ParagraphStyle("H", parent=base, fontName="DejaVuBold", fontSize=10.6, leading=12.5, spaceBefore=4, spaceAfter=3, textColor=colors.HexColor("#0b2f4f"))
small = ParagraphStyle("Small", parent=base, fontSize=8.1, leading=9.6)

accent = colors.HexColor("#1f4e79")
light_accent = colors.HexColor("#e8f0fb")
grey = colors.HexColor("#444444")

def header_footer(canvas, doc):
    canvas.saveState()
    w, hgt = A4
    bar_h = 20*mm
    canvas.setFillColor(light_accent)
    canvas.rect(0, hgt-bar_h, w, bar_h, stroke=0, fill=1)
    canvas.setFillColor(accent)
    canvas.setFont("DejaVuBold", 11)
    canvas.drawString(12*mm, hgt-8.5*mm, "Naamgeving organische moleculen (C1–C5)")
    canvas.setFont("DejaVu", 8)
    canvas.setFillColor(grey)
    canvas.drawString(12*mm, hgt-15.5*mm, "Koolwaterstoffen, halogeenalkanen, alcoholen, carbonzuren (1 functionele groep)")
    canvas.setFillColor(grey)
    canvas.setFont("DejaVu", 8)
    canvas.drawString(12*mm, 8*mm, "Drill: structuur → naam én naam → structuur. Focus op locanten en prioriteit.")
    canvas.drawRightString(w-12*mm, 8*mm, f"Pagina {doc.page}")
    canvas.restoreState()

# Bestandspad op macOS (in Downloads of huismap)
output_dir = Path.home() / "Documents"
file_path = str(output_dir / "A4_Cheatsheet_Naamgeving_C1-C5.pdf")
print(f"PDF naar: {file_path}")

doc = BaseDocTemplate(file_path, pagesize=A4, leftMargin=12*mm, rightMargin=12*mm, topMargin=24*mm, bottomMargin=14*mm)

w, hgt = A4
usable_w = w - doc.leftMargin - doc.rightMargin
gutter = 6*mm
col_w = (usable_w - gutter) / 2
frame_h = hgt - doc.topMargin - doc.bottomMargin
frame1 = Frame(doc.leftMargin, doc.bottomMargin, col_w, frame_h, id="col1", showBoundary=0)
frame2 = Frame(doc.leftMargin + col_w + gutter, doc.bottomMargin, col_w, frame_h, id="col2", showBoundary=0)
doc.addPageTemplates([PageTemplate(id="TwoCol", frames=[frame1, frame2], onPage=header_footer)])

story = [Spacer(1,2), Paragraph("Cheat sheet + 25 oefenitems (met oplossingen)", title)]

story.append(Paragraph("1) Mini-woordenboek", h))
story.append(Paragraph("<b>Stammen</b>: meth- (1), eth- (2), prop- (3), but- (4), pent- (5).", base))
story.append(Paragraph("<b>Verzadiging</b>: -aan (enkel), -een (dubbel), -yn (drievoudig). Locant bij binding: but-1-een, pent-2-yn.", base))
story.append(Paragraph("<b>Substituenten</b>: methyl- (CH3), ethyl- (C2H5); halogenen: fluor-, chloor-, broom-, jood-. Meervoud: di-, tri-.", base))
story.append(Paragraph("<b>Functie (1 per molecule)</b>: alcohol = -ol; carbonzuur = -zuur.", base))

story.append(Paragraph("2) Prioriteit en nummering", h))
story.append(Paragraph("<b>carbonzuur</b> (COOH) &gt; <b>alcohol</b> (OH) &gt; dubbele/drievoudige binding &gt; substituenten.", base))
story.append(Paragraph("Nummer zodat de <b>laagste locanten</b> ontstaan (first point of difference).", base))

story.append(Paragraph("3) Stappenplan (structuur → naam)", h))
steps = [
    "Kies de <b>langste keten</b> die de functie/binding bevat.",
    "Kies het <b>suffix</b>: -zuur, -ol, -een, -yn, -aan.",
    "Nummer: functie → binding → substituenten.",
    "Schrijf substituenten met locanten, <b>alfabetisch</b> (di/tri tellen niet mee).",
    "Combineer: locanten–prefixen + stam + (locant binding) + suffix."
]
story.append(Paragraph("<br/>".join([f"• {s}" for s in steps]), base))

story.append(Paragraph("4) Klassieke valkuilen", h))
pitfalls = [
    "Di-/tri- <b>niet</b> meetellen voor alfabet (dichloor- blijft onder 'chloor').",
    "Hoofdketen te kort: neem echt de langste die de functie/binding bevat.",
    "Isomeren: 1-chloorpropaan ≠ 2-chloorpropaan; but-1-een ≠ but-2-een.",
    "Locanten vergeten bij alkenen/alkynen: but-1-een is iets anders dan but-2-een."
]
story.append(Paragraph("<br/>".join([f"• {p}" for p in pitfalls]), base))

story.append(Paragraph("5) Ankers (ken deze blind)", h))
anchors = [
    "<b>C3</b>: propaan; prop-1-een; prop-1-yn; propan-1-ol; propan-2-ol; propaanzuur.",
    "<b>C4</b>: butaan; 2-methylpropaan; but-1-een; but-2-een; butan-1-ol; butan-2-ol; butaanzuur.",
    "<b>C5</b>: pentaan; 2-methylbutaan; 2,2-dimethylpropaan; pent-2-een; pentaanzuur.",
]
story.append(Paragraph("<br/>".join([f"• {a}" for a in anchors]), base))

story.append(Paragraph("6) Minimal pairs (snelle check)", h))
mp = [
    "propan-1-ol vs propan-2-ol",
    "1,1-dichloorethaan vs 1,2-dichloorethaan",
    "but-1-een vs but-2-een",
    "2-methylbutaan vs 2,2-dimethylpropaan"
]
story.append(Paragraph("<br/>".join([f"• {m}" for m in mp]), base))

story.append(FrameBreak())

story.append(Paragraph("7) Oefenen (25) — met oplossingen", h))
story.append(Paragraph("Mengvorm: soms structuur → naam, soms naam → structuur. Antwoord staat rechts.", small))
story.append(Spacer(1,4))

ex = [
 ("CH3–CH2–CH3", "propaan"),
 ("CH3–CH2–CH2–CH3", "butaan"),
 ("CH3–CH(CH3)–CH3", "2-methylpropaan"),
 ("CH3–CH2–CH2–CH2–CH3", "pentaan"),
 ("CH3–C(CH3)2–CH3", "2,2-dimethylpropaan"),
 ("CH2=CH–CH3", "prop-1-een (propeen)"),
 ("CH3–CH=CH–CH3", "but-2-een"),
 ("CH≡C–CH3", "prop-1-yn (propyn)"),
 ("CH3–CH2–C≡CH", "but-1-yn"),
 ("CH3–CH2–CH(OH)–CH3", "butan-2-ol"),
 ("CH3–CH(OH)–CH3", "propan-2-ol (isopropanol)"),
 ("HO–CH2–CH2–CH3", "propan-1-ol"),
 ("CH3–CH2–COOH", "propaanzuur"),
 ("CH3–COOH", "ethaanzuur (azijnzuur)"),
 ("(CH3)2CH–COOH", "2-methylpropaanzuur"),
 ("CH3–CH2–CH2–Cl", "1-chloorpropaan"),
 ("CH3–CH(Cl)–CH3", "2-chloorpropaan"),
 ("CH3–CH2–CH(Br)–CH3", "2-broombutaan"),
 ("CH3–CH(Cl)–CH2–CH3", "2-chloorbutaan"),
 ("CH3–CH2–CH2–CH2–I", "1-joodbutaan"),
 ("1,2-dichloorethaan", "Cl–CH2–CH2–Cl"),
 ("1,1-dichloorethaan", "CH3–CHCl2"),
 ("2-methylbutaan", "CH3–CH(CH3)–CH2–CH3"),
 ("pent-2-een", "CH3–CH=CH–CH2–CH3"),
 ("2,3-dibroompentaan", "CH3–CH(Br)–CH(Br)–CH2–CH3"),
]

def make_table(items, font=7.8):
    data = [["Oefening", "Antwoord / structuur"]]
    for q, a in items:
        data.append([q, a])
    tbl = Table(data, colWidths=[col_w*0.52, col_w*0.48])
    tbl.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,0), "DejaVuBold"),
        ("FONTSIZE", (0,0), (-1,0), 8.2),
        ("BACKGROUND", (0,0), (-1,0), light_accent),
        ("TEXTCOLOR", (0,0), (-1,0), accent),
        ("FONTNAME", (0,1), (-1,-1), "DejaVu"),
        ("FONTSIZE", (0,1), (-1,-1), font),
        ("LEADING", (0,1), (-1,-1), font+1.2),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.whitesmoke, colors.white]),
        ("GRID", (0,0), (-1,-1), 0.25, colors.lightgrey),
        ("LEFTPADDING", (0,0), (-1,-1), 4),
        ("RIGHTPADDING", (0,0), (-1,-1), 4),
        ("TOPPADDING", (0,0), (-1,-1), 2),
        ("BOTTOMPADDING", (0,0), (-1,-1), 2),
    ]))
    return tbl

story.append(make_table(ex[:13]))
story.append(Spacer(1,6))
story.append(make_table(ex[13:]))
story.append(Spacer(1,6))
story.append(Paragraph("<b>Bonus:</b> voeg 5 eigen kaarten toe uit oefeningen/oud-examens.", small))

doc.build(story)
print(f"✓ PDF gegenereerd: {file_path}")
file_path

⚠ Fonts niet gevonden. Fallback naar standaard fonts.
PDF naar: /Users/tristancools/Documents/A4_Cheatsheet_Naamgeving_C1-C5.pdf


ValueError: 
paragraph text '<para>Cheat sheet + 25 oefenitems (met oplossingen)</para>' caused exception error with style name=Title Can't map determine family/bold/italic for dejavubold